# TimesFM 3 × FEV input-strategy smoke

Mirrors the verified Chronos-2 smoke on one **calibration** origin of `epf_np`. It uses the pinned official FEV TimesFM-3 wrapper and records the exact model revision. Results are `smoke_only`.

The TimesFM 3 weights use the **TimesFM Non-Commercial License v1.0** and are used here only for academic, non-commercial research. Select **T4 GPU** before running all cells.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/covariate-safe-tsfm')
FEV_CHECKOUT = Path('/content/fev-pinned')
FEV_COMMIT = '38007871dcf6dc6b04aed3a54d9cd86678d48d0b'
TIMESFM_COMMIT = '20191171b74f51bfead932b6b8d0c8f515e70f63'

if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/FlyMe2star/covariate-safe-tsfm.git', str(REPO)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

if not FEV_CHECKOUT.exists():
    subprocess.run(
        ['git', 'clone', '--filter=blob:none',
         'https://github.com/autogluon/fev.git', str(FEV_CHECKOUT)],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(FEV_CHECKOUT), 'checkout', FEV_COMMIT],
        check=True,
    )
else:
    observed_fev_commit = subprocess.check_output(
        ['git', '-C', str(FEV_CHECKOUT), 'rev-parse', 'HEAD'], text=True
    ).strip()
    assert observed_fev_commit == FEV_COMMIT, observed_fev_commit

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)],
    check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        f'git+https://github.com/autogluon/fev.git@{FEV_COMMIT}',
        ('timesfm[torch] @ git+https://github.com/google-research/'
         f'timesfm.git@{TIMESFM_COMMIT}'),
    ],
    check=True,
)

SOURCE_ROOT = str(REPO / 'src')
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
importlib.invalidate_caches()
covsafe = importlib.import_module('covsafe')
print('Repository ready:', REPO)
print('Pinned FEV checkout:', FEV_CHECKOUT)
print('covsafe import:', covsafe.__file__)

In [ ]:
import torch

assert torch.cuda.is_available(), (
    'Select Runtime > Change runtime type > T4 GPU, then restart.'
)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('Torch:', torch.__version__)

In [ ]:
import json

from covsafe.timesfm3_smoke import (
    EXPECTED_SMOKE_CONFIG_HASH,
    run_timesfm3_smoke,
)

print('Frozen smoke config hash:', EXPECTED_SMOKE_CONFIG_HASH)
manifest = run_timesfm3_smoke(REPO, FEV_CHECKOUT)
print(json.dumps(manifest, indent=2, ensure_ascii=False, default=str))

In [ ]:
import shutil

from google.colab import drive

drive.mount('/content/drive', force_remount=False)
local_manifest = REPO / 'outputs/smoke/timesfm3_epf_np.json'
drive_manifest = (
    Path('/content/drive/MyDrive/covariate-safe-tsfm/private_manifests')
    / 'timesfm3_epf_np_smoke.json'
)
drive_manifest.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(local_manifest, drive_manifest)
print('Durable manifest:', drive_manifest)

## Return artifact

Send the JSON printed by the third code cell. A valid manifest reports both variants, finite SQL/WQL/MASE/WAPE, the checkpoint and source revisions, `scientific_gate_computed: false`, and `sealed_evaluation_origins_instantiated: false`.